In [ ]:
!pip install -q simpletransformers
!pip install -q transformers
!pip install -q torch
!pip install -q scikit-learn
!pip install -q pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.5/332.5 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 140.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 129.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd

df = pd.read_csv("mtsamples.csv")

print(df.shape)

df.head()

(4999, 6)


,Unnamed: 0,description,medical_specialty,sample_name,transcription,keywords
0,0,A 23-year-old white female presents with comp...,Allergy / Immunology,Allergic Rhinitis,"SUBJECTIVE:, This 23-year-old white female pr...","allergy / immunology, allergic rhinitis, aller..."
1,1,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 2,"PAST MEDICAL HISTORY:, He has difficulty climb...","bariatrics, laparoscopic gastric bypass, weigh..."
2,2,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 1,"HISTORY OF PRESENT ILLNESS: , I have seen ABC ...","bariatrics, laparoscopic gastric bypass, heart..."
3,3,2-D M-Mode. Doppler.,Cardiovascular / Pulmonary,2-D Echocardiogram - 1,"2-D M-MODE: , ,1. Left atrial enlargement wit...","cardiovascular / pulmonary, 2-d m-mode, dopple..."
4,4,2-D Echocardiogram,Cardiovascular / Pulmonary,2-D Echocardiogram - 2,1. The left ventricular cavity size and wall ...,"cardiovascular / pulmonary, 2-d, doppler, echo..."


In [ ]:
df = df.dropna(subset=["transcription"])

df["keywords"] = df["keywords"].fillna("")

df["text"] = (
    df["keywords"]
    + " "
    + df["transcription"]
)

print(df.shape)

(4966, 7)


In [ ]:
counts = df["medical_specialty"].value_counts()

rare_classes = counts[counts < 30].index

df["medical_specialty"] = df["medical_specialty"].replace(
    rare_classes,
    "Other"
)

print(df["medical_specialty"].value_counts())

medical_specialty
 Surgery                          1088
 Consult - History and Phy.        516
 Cardiovascular / Pulmonary        371
 Orthopedic                        355
 Radiology                         273
Other                              272
 General Medicine                  259
 Gastroenterology                  224
 Neurology                         223
 SOAP / Chart / Progress Notes     166
 Urology                           156
 Obstetrics / Gynecology           155
 Discharge Summary                 108
 ENT - Otolaryngology               96
 Neurosurgery                       94
 Hematology - Oncology              90
 Ophthalmology                      83
 Nephrology                         81
 Emergency Room Reports             75
 Pediatrics - Neonatal              70
 Pain Management                    61
 Psychiatry / Psychology            53
 Office Notes                       50
 Podiatry                           47
Name: count, dtype: int64


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["labels"] = le.fit_transform(
    df["medical_specialty"]
)

print("Number of classes:",
      df["labels"].nunique())

Number of classes: 24


In [ ]:
from sklearn.model_selection import train_test_split

train_df, eval_df = train_test_split(
    df[["text","labels"]],
    test_size=0.2,
    random_state=42,
    stratify=df["labels"]
)

print(train_df.shape)
print(eval_df.shape)

print(df["labels"].nunique())

(3972, 2)
(994, 2)
24


In [ ]:
from simpletransformers.classification import (
    ClassificationModel,
    ClassificationArgs
)

model_args = ClassificationArgs()

model_args.num_train_epochs = 2
model_args.train_batch_size = 8
model_args.eval_batch_size = 8

model_args.overwrite_output_dir = True

model_args.save_model_every_epoch = False
model_args.save_eval_checkpoints = False
model_args.save_steps = -1

model = ClassificationModel(
     model_type="bert",
    model_name="dmis-lab/biobert-base-cased-v1.1",
    num_labels=df["labels"].nunique(),
    args=model_args,
    use_cuda=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

In [ ]:
model.train_model(train_df)

  0%|          | 0/7 [00:00<?, ?it/s]

Epoch:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:924: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler()


Running Epoch 1 of 2:   0%|          | 0/497 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:950: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():


Running Epoch 2 of 2:   0%|          | 0/497 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

(994, 0.8519631024577489)

In [ ]:
result, model_outputs, wrong_predictions = model.eval_model(
    eval_df
)

print(result)

  0%|          | 0/1 [00:00<?, ?it/s]

Running Evaluation:   0%|          | 0/125 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:1570: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():


{'mcc': np.float64(0.8202598968423034), 'eval_loss': 0.4921985607147217}


In [ ]:
from sklearn.metrics import accuracy_score

predictions, raw_outputs = model.predict(eval_df["text"].tolist())

accuracy = accuracy_score(
    eval_df["labels"],
    predictions
)

print("Accuracy:", accuracy)

  0%|          | 0/1 [00:00<?, ?it/s]

Predicting:   0%|          | 0/125 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:2260: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():


Accuracy: 0.8360160965794768


In [ ]:
sample_text = eval_df.iloc[0]["text"]

predictions, raw_outputs = model.predict(
    [sample_text]
)

print("Predicted:",
      le.inverse_transform(predictions)[0])

print()

print("Actual:",
      le.inverse_transform(
          [eval_df.iloc[0]["labels"]]
      )[0])

0it [00:00, ?it/s]

Predicting:   0%|          | 0/1 [00:00<?, ?it/s]

Predicted:  Radiology

Actual:  Radiology


/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:2260: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():


In [ ]:
!du -sh outputs
!ls -lh outputs

414M	outputs
total 414M
-rw-r--r-- 1 root root 1.8K Jun 22 00:23 config.json
-rw-r--r-- 1 root root   56 Jun 22 00:23 eval_results.txt
-rw-r--r-- 1 root root 2.9K Jun 22 00:23 model_args.json
-rw------- 1 root root 414M Jun 22 00:23 model.safetensors
-rw-r--r-- 1 root root  379 Jun 22 00:23 tokenizer_config.json
-rw-r--r-- 1 root root 654K Jun 22 00:23 tokenizer.json
-rw-r--r-- 1 root root 4.3K Jun 22 00:23 training_args.bin


In [ ]:

from google.colab import files
!zip -r outputs.zip outputs
files.download("outputs.zip")

  adding: outputs/ (stored 0%)
  adding: outputs/model.safetensors (deflated 7%)
  adding: outputs/config.json (deflated 65%)
  adding: outputs/tokenizer_config.json (deflated 43%)
  adding: outputs/eval_results.txt (deflated 2%)
  adding: outputs/tokenizer.json (deflated 70%)
  adding: outputs/training_args.bin (deflated 53%)
  adding: outputs/model_args.json (deflated 61%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import joblib

joblib.dump(le, "label_encoder.pkl")
files.download("label_encoder.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json

label_mapping = {
    int(i): label
    for i, label in enumerate(le.classes_)
}

with open("label_mapping.json", "w") as f:
    json.dump(label_mapping, f, indent=4)

files.download("label_mapping.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
sample_text = """
The patient presented with severe headaches and
episodes of confusion. MRI showed temporal lobe
abnormalities.
"""

predictions, raw_outputs = model.predict([sample_text])

predicted_specialty = le.inverse_transform(predictions)[0]

print(predicted_specialty)

0it [00:00, ?it/s]

Predicting:   0%|          | 0/1 [00:00<?, ?it/s]

 Neurology


/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:2260: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():
